# 3. Streamer tiers

Bucket streamers into small, mid-size, and mega tiers by typical concurrent viewers, then compare engagement lift per viewer-hour across tiers.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
COLLECTED = ROOT / "data" / "collected"
con = duckdb.connect()

In [ ]:
twitch = con.sql(f"SELECT * FROM read_csv_auto('{COLLECTED}/**/twitch_*.csv', union_by_name=true)").df()
twitch["snapshot_utc"] = pd.to_datetime(twitch["snapshot_utc"])

In [ ]:
# Typical audience per streamer (median concurrent viewers across snapshots)
by_streamer = twitch.groupby('user_login').viewer_count.median()
tiers = pd.cut(by_streamer, [0, 1_000, 10_000, float('inf')], labels=['small', 'mid', 'mega'])
tiers.value_counts()

In [ ]:
# Viewer-hours per game-hour by tier (each snapshot ≈ one hour)
twitch['tier'] = twitch.user_login.map(tiers)
vh = twitch.pivot_table(index=['steam_appid', 'snapshot_utc'], columns='tier', values='viewer_count', aggfunc='sum', fill_value=0)
vh.head()